# 🧹 Phase 1: Data Cleaning & Pre-processing

This notebook implements a high-performance, **parallelized cleaning pipeline** designed to handle large anime datasets efficiently.

### 🚀 Key Optimization Strategies
-   **Atomic Pipelines**: Each dataset is processed by a specialized, modular pipeline.
-   **Parallel Processing**: CPU-intensive tasks are distributed across all available cores.
-   **Memory Management**: Aggressive downcasting and categorical conversion typically reduce memory usage by **80%+**.
-   **Referential Integrity**: A progressive `GLOBAL_ID_MAP` ensures all datasets link to valid, canonical IDs.


In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
from joblib import Parallel, delayed
import pandas as pd

# 🧪 Standard Imports
from solution.src.common import PROCESSED_DATA_DIR
from solution.src.data import load_raw, get_raw_path, PROCESSED_FILES
from solution.src.utils import save_refined, print_fancy_progress

# 📦 Cleaning Utilities
from nb_01_cleaning import (
    REMAINING_FILES,
    standardize_column_names_and_text,
    trim_all_text_columns,
    merge_content_duplicates,
    remove_orphaned_records,
    get_memory_usage_mb,
    reduce_memory_usage,
    display_cleaning_summary,
    display_foreign_key_merges,
    process_ratings_batch,
    clean_and_optimize_dataset,
    count_total_rows_in_file,
    clean_characters_dataset,
    clean_anime_details_dataset,
    clean_anime_stats_dataset
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print(f"Saving to:    {PROCESSED_DATA_DIR}")

# Global ID Map (relational integrity)
GLOBAL_ID_MAP = {}


## 1. Characters (`characters.csv`)

**Role**: Establishes the valid universe of Character IDs.

**Processing Steps**:
-   **Deduplication**: Identifies and merges duplicate characters, updating the Global ID Map.
-   **Cleaning**: Drops `about` sections to save space (they're not important for our needs).
-   **Validation**: Captures `valid_character_ids` to filter orphan references in other files.

In [ ]:
# 1. Load
df_chars = load_raw('characters')

# 2. Clean Pipeline
df_chars, id_map, report = clean_characters_dataset(df_chars)

# 3. Update Global Knowledge
if id_map:
    GLOBAL_ID_MAP['character_id'] = id_map

# 4. Report
display_cleaning_summary("Characters", report)

# 5. Optimize & Save
df_chars = reduce_memory_usage(df_chars)
save_refined(df_chars, PROCESSED_DATA_DIR / "characters.parquet")

# 6. Context Capture
valid_character_ids = set(df_chars['character_id'].dropna())
print(f"Captured {len(valid_character_ids)} valid Character IDs.")

gc.collect();


## 2. Anime Details (`details.csv`)

**Role**: Establishes the valid universe of Anime IDs.

**Processing Steps**:
-   **Imputation**: Derives missing `year` from `start_date`.
-   **Type Conversion**: Parses dates and converts low-cardinality columns (Genres, Studios) to Categories.
-   **Validation**: Captures `valid_anime_ids`.

In [ ]:
# 1. Load
df_details = load_raw('details')

# 2. Clean Pipeline
df_details, id_map, report = clean_anime_details_dataset(df_details)

# 3. Update Global Knowledge
if id_map:
    GLOBAL_ID_MAP['anime_id'] = id_map

# 4. Report
display_cleaning_summary("Details", report)

# 5. Optimize & Save
df_details = reduce_memory_usage(df_details)
save_refined(df_details, PROCESSED_DATA_DIR / "details.parquet")

# 6. Context Capture
valid_anime_ids = set(df_details['anime_id'].dropna())
print(f"Captured {len(valid_anime_ids)} valid Anime IDs.")

gc.collect();


## 3. Anime Stats (`stats.csv`)

**Role**: Supplementary validation source. Often contains duplicates not present in `details.csv`.

**Processing Steps**:
-   **Merge Duplicates**: Updates the `GLOBAL_ID_MAP` if new duplicates are found.
-   **Referential Integrity**: Ensures all entries map to valid Anime IDs from Step 2.

In [ ]:
# 1. Load
df_stats = load_raw('stats')

# 2. Clean Pipeline
df_stats, id_map, report = clean_anime_stats_dataset(df_stats)

# 3. Update Global Knowledge (Merge if new duplicates found)
if id_map:
    if 'anime_id' not in GLOBAL_ID_MAP:
        GLOBAL_ID_MAP['anime_id'] = {}
    GLOBAL_ID_MAP['anime_id'].update(id_map)

# 4. Report
display_cleaning_summary("Stats", report)

# 5. Optimize & Save
df_stats = reduce_memory_usage(df_stats)
save_refined(df_stats, PROCESSED_DATA_DIR / "stats.parquet")

gc.collect();


## 4. User Ratings (`ratings.csv`)

**Challenge**: Huge dataset (>120M rows). Cannot process in memory sequentially without being slow.

**Solution: Parallel Streaming Approach**
-   **Parallel Workers**: The file is read in chunks, and each chunk is processed by a separate CPU core.
-   **Streaming Write**: Processed chunks are immediately written to disk (Parquet), keeping RAM usage low.
-   **FK Remapping**: We apply the `GLOBAL_ID_MAP` here to update Anime IDs that were merged in previous steps. 
-   **Orphan Removal**: Votes for non-existent Anime are dropped.

In [ ]:
chunk_size = 5_000_000
csv_path = get_raw_path('ratings')

print(f"Processing {csv_path} in parallel chunks...")

# Context for workers
id_map_anime = GLOBAL_ID_MAP.get('anime_id', {})

stats = {'rows_read': 0, 'dropped_orphans': 0, 'fk_remapped': 0, 'fk_samples': []}

print("Counting rows for progress estimation...")
total_rows = count_total_rows_in_file(csv_path) - 1
estimated_total_chunks = (total_rows // chunk_size) + (1 if total_rows % chunk_size > 0 else 0)
print(f"Target: {total_rows:,} rows across {estimated_total_chunks} chunks.")

# Parallel Pipeline Generator
reader = pd.read_csv(csv_path, chunksize=chunk_size, engine='c')
processed_chunks = Parallel(n_jobs=-1, return_as='generator')(
    delayed(process_ratings_batch)(chunk, valid_anime_ids, id_map_anime)
    for chunk in reader
)

# Streaming Write
def result_generator():
    for i, (processed_chunk, metrics) in enumerate(processed_chunks, 1):
        print_fancy_progress(i, estimated_total_chunks, f'Chunk {i}: ')

        stats['rows_read'] += metrics['rows_read']
        stats['dropped_orphans'] += metrics['dropped_orphans']
        stats['fk_remapped'] += metrics['fk_remapped']
        if len(stats['fk_samples']) < 5:
            stats['fk_samples'].extend(metrics['fk_samples'])

        yield processed_chunk

save_refined(result_generator(), PROCESSED_DATA_DIR / "ratings.parquet")

# Final Report
final_report = {
    "Total Rows Read": stats['rows_read'],
    "Orphans Dropped": stats['dropped_orphans'],
    "FKs Remapped": stats['fk_remapped'],
    "Final Rows Est.": stats['rows_read'] - stats['dropped_orphans']
}
display_cleaning_summary("Ratings", final_report)
display_foreign_key_merges(stats['fk_samples'])

gc.collect();


## 5. Remaining Files

**Strategy**: Apply the same rigor to the remaining 9 Relational and Metadata tables.

**Parallel Execution**:
We use `clean_and_optimize_dataset` (defined in `nb_01_cleaning` package) to run the cleaning pipeline for all these files **concurrently**. 

**Understanding the Report:**
-   **`Clean` Impact**: Memory usage usually *increases* here (e.g., `-10%` reduction) because we expand efficient storage into Python objects for cleaning.
-   **`Opt` Impact**: Memory usage *decreases* massively (e.g., `+80%` reduction) as we convert to efficient types.
-   **`Total` Impact**: The net result. Usually a significant reduction, ensuring the final Parquet files are highly optimized.

In [ ]:
print(f"Processing {len(REMAINING_FILES)} other files in parallel...")

results = Parallel(n_jobs=-1)(
    delayed(clean_and_optimize_dataset)(name, PROCESSED_FILES)
    for name in REMAINING_FILES
)

print(f"\n--- Detailed Batch Report ---")
for res in results:
    total_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0

    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.1f}MB -> {res['mem_dst']:.1f}MB ({total_red * -1:+.1f}%)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)

print(f"Total files processed: {len(results)}")

gc.collect();
